# Osonye Onyemazuwa — Ad Spend Data
### Day 1-2: Created main branch README + Setup issues on Github + Acquire and load Ad Spend dataset

Data file: `ad_spend.csv` (place in a local `data/` folder, excluded via `.gitignore`)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("ad_spend.csv")
print(df.shape)
df.head()


## Initial structure check

In [ ]:
df.dtypes

In [ ]:
df.isna().sum()

In [ ]:
df.describe(include='all').T

In [ ]:
# Convert date from string to a real datetime type
df['date'] = pd.to_datetime(df['date'])
df.dtypes

## Quick look: spend by channel

In [ ]:
df.groupby('channel')['ad_spend'].agg(['sum', 'mean', 'count']).sort_values('sum', ascending=False)


## Quick look: date range

In [ ]:
print("Earliest:", df['date'].min())
print("Latest:", df['date'].max())


## Notes (local analysis log)

- Rows: 2,599, no missing values in any column (confirmed clean on delivery)
- Date range: 2021-01-20 to 2024-01-06
- `campaign_id` is numeric, joins cleanly to `campaigns.csv` (all 50 IDs used, no orphans either direction)
- No duplicate `spend_id` rows
- Spend by channel: **Affiliate (\$6,455)** > **Paid Search (\$6,151)** > **Email (\$5,614)** \> **Display (\$4,480)** \> **Social (\$4,176)**

Team-facing decisions/blockers from this analysis are tracked in the repo README, not duplicated here.


---
### Next steps (Day 3 — Issue #3)
- Standardize `channel` / `utm_source` casing to match `events.traffic_source`
- Decide with team how to handle Affiliate/Display spend attribution
- Check for duplicate `spend_id` rows


---
## Day 3 — Clean campaign IDs and standardize channel naming
**Issue #8:** Clean missing UTM parameters and inconsistent campaign IDs

Checked already:
- `campaign_id`: all 50 IDs match `campaigns.csv` exactly (no orphans either direction) — no cleaning needed here
- `spend_id`: no duplicates — no cleaning needed here
- No missing values anywhere in this file

Remaining real issue: `channel` values here don't match `events.traffic_source`
casing/naming, which will break the Week 2 SQL joins if not fixed now.


In [ ]:
# Confirm campaign_id integrity (already checked, keeping as a documented test)
campaigns = pd.read_csv("campaigns.csv")

ad_ids = set(df['campaign_id'])
valid_ids = set(campaigns['campaign_id'])

orphans_in_spend = ad_ids - valid_ids
unused_campaigns = valid_ids - ad_ids

print("campaign_ids in ad_spend but not in campaigns.csv:", orphans_in_spend)
print("campaign_ids in campaigns.csv but never spent on:", unused_campaigns)
assert len(orphans_in_spend) == 0, "Found orphan campaign_ids -- investigate before Week 2"

In [ ]:
# Confirm no duplicate spend_id rows
dupes = df['spend_id'].duplicated().sum()
print(f"Duplicate spend_id rows: {dupes}")
assert dupes == 0, "Found duplicate spend_id rows -- dedupe before proceeding"

### Standardize `channel` for downstream joins

`events.traffic_source` uses: Direct / Email / Organic / Paid Search / Social
(in multiple casings). `ad_spend.channel` uses: Affiliate / Display / Email /
Paid Search / Social.

Standardizing to lowercase + underscores now so this matches whatever
Member B does to `events.traffic_source` in their own cleaning step —
**worth confirming the exact convention with Member B before Day 4**,
so both sides land on the same format.

In [ ]:
df['channel_clean'] = df['channel'].str.strip().str.lower().str.replace(' ', '_', regex=False)
df['utm_source_clean'] = df['utm_source'].str.strip().str.lower()

print(df['channel_clean'].unique())
print(df['utm_source_clean'].unique())

### Flag: Affiliate / Display have no equivalent in `events.traffic_source`

> This isn't something Day 3 can "clean away" — it's a team decision on how
> Affiliate and Display spend gets attributed (map to an existing category,
> or keep as a separate "unattributed spend" line). Documented in README;
> raising with team before Week 3 Fact table work.

In [ ]:
# Save cleaned output locally (not committed -- .gitignore covers this)
import os
os.makedirs("cleaned", exist_ok=True)
df.to_csv("cleaned/ad_spend_clean.csv", index=False)
print("Saved cleaned/ad_spend_clean.csv:", df.shape)

---
## Day 4 — EDA on Ad Spend: spend by channel/campaign/day
**Issue #9:** EDA on Ad Spend: spend by channel/campaign/day; commit charts + notes.

In [ ]:
df.head()

### Spend by channel

In [ ]:
spend_by_channel = df.groupby('channel_clean')['ad_spend'].sum().sort_values(ascending=False)
spend_by_channel

In [ ]:
spend_by_channel.plot(kind='bar', figsize=(8,4), title='Total Ad Spend by Channel')
plt.ylabel('Spend ($)')
plt.tight_layout()
plt.savefig('spend_by_channel.png')
plt.show()